## Mullvad Browser Docker Setup

To run **Mullvad Browser** inside an isolated Docker container (useful for private browsing), start it with the following command:

```bash
docker run -d \
  --name mullvad-browser \
  --platform=linux/amd64 \
  -p 3000:3000 \
  -p 3001:3001 \
  -e TZ=Asia/Ho_Chi_Minh \
  -v "$(pwd)/mullvadData:/config/Downloads" \
  --shm-size=1gb \
  lscr.io/linuxserver/mullvad-browser:latest
```


# Pipeline

            ┌───────────────────────────────┐
            │        Input Company Name     │
            │         (e.g. Toyota...)      │
            └───────────────┬───────────────┘
                            │
                            ▼
            ┌───────────────────────────────┐
            │ 1. Search Query Execution     │
            │   (Playwright Automation)     │
            │ - Open Mullvad Browser        │
            │ - Type: site:spglobal.com ... │
            │ - Submit search request       │
            └───────────────┬───────────────┘
                            │
                            ▼
            ┌───────────────────────────────┐
            │ 2. Search Results Capture     │
            │   (Clipboard Copy)            │
            │ - Select all SERP text        │
            │ - Copy results into clipboard │
            │ - Extract raw search output   │
            └───────────────┬───────────────┘
                            │
                            ▼
            ┌───────────────────────────────┐
            │ 3. Best ESG URL Selection     │
            │   (Gemini Single Call)        │
            │ - Extract ALL titles + URLs   │
            │ - Normalize url.              │
            │ - Pick best ESG Score page    │
            │ - Return JSON result list     │
            └───────────────┬───────────────┘
                            │
                            ▼
            ┌───────────────────────────────┐
            │ 4. ESG Page Source Loading    │
            │   (view-source Navigation)    │
            │ - Open: view-source:<URL>     │
            │ - Wait full HTML rendering    │
            │ - Copy entire page source     │
            └───────────────┬───────────────┘
                            │
                            ▼
            ┌───────────────────────────────┐
            │ 5. ESG Table Extraction       │
            │   (BeautifulSoup Parser)      │
            │ - Locate esg-detail-table     │
            │ - Support Desktop + Mobile UI │
            │ - Extract headers + values    │
            │ - Convert into Markdown table │
            └───────────────┬───────────────┘
                            │
                            ▼
            ┌───────────────────────────────┐
            │ 6. Final Structured Output    │
            │   (Row Data per Company)      │
            │ - Best ESG URL saved          │
            │ - Markdown ESG result stored  │
            │ - Export-ready for CSV file   │
            └───────────────────────────────┘

In [8]:
import asyncio
import os
import shutil
from pathlib import Path
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import pyperclip
import google.generativeai as genai
import json
import re
import csv
import unicodedata
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

SEARCH_URL = "http://localhost:3000"
SOURCE_DIR = Path("mullvadData")
CSV_FILENAME = "esg_results.csv"

class SearchAndSelectURL:
    """Extract search results and select best URL in ONE Gemini call"""
    
    def __init__(self, model_name="gemini-2.5-flash"):
        self.model_name = model_name

    def run_gemini(self, prompt: str) -> str:
        try:
            model = genai.GenerativeModel(model_name=self.model_name)
            response = model.generate_content(
                prompt,
                generation_config={
                    "temperature": 0,
                    "response_mime_type": "application/json",
                },
            )
            if not response or not hasattr(response, 'text') or not response.text:
                print(f"Empty response from Gemini")
                return '{"title": "", "url": "", "all_results": []}'
            return response.text.strip()
        except Exception as e:
            print(f"Gemini API Error: {e}")
            return '{"title": "", "url": "", "all_results": []}'

    def normalize_url(self, url: str) -> str:
        url = url.strip()
        if "›" in url:
            url = url.replace("›", "/")
            url = re.sub(r"\s+", "", url)
        return url

    def select_best_url(self, company_name: str, copied_text: str) -> dict:
        """
        Extract results AND select best URL in ONE call.
        Returns: {
            "title": "...",
            "url": "...",
            "all_results": [{"title": "...", "url": "..."}, ...]
        }
        """
        if not copied_text or copied_text.strip() == "":
            print("No search results copied from browser")
            return {"title": "", "url": "", "all_results": []}
        
        prompt = f"""
        You are given copied text from a search engine results page.

        Company: {company_name}
        Query: site:spglobal.com {company_name} ESG Score

        Task:
        1. Extract ALL search results (title + url)
        2. Identify the SINGLE BEST result matching the ESG Score page for this company
        3. Prioritize spglobal.com results

        Return ONLY JSON:
        {{
        "title": "<best result title>",
        "url": "<best result url>",
        "all_results": [
            {{"title": "...", "url": "..."}},
            ...
        ]
        }}

        Rules:
        - Ignore ads/navigation
        - URLs may have › symbols (convert to /)
        - best_url should be the URL of the top ESG score result
        - Include ALL extracted results in all_results array

        Copied text:
        ----------------
        {copied_text}
        ----------------
        """
        try:
            raw_json = self.run_gemini(prompt)
            if not raw_json or raw_json.strip() == "":
                return {"title": "", "url": "", "all_results": []}
            
            data = json.loads(raw_json)

            # Normalize URLs
            if data.get("url"):
                data["url"] = self.normalize_url(data["url"])
            for r in data.get("all_results", []):
                if r.get("url"):
                    r["url"] = self.normalize_url(r["url"])

            return data
        except json.JSONDecodeError as e:
            print(f"JSON Parse Error: {e}")
            print(f"   Raw response: {raw_json[:200]}")
            return {"title": "", "url": "", "all_results": []}
        except Exception as e:
            print(f"Error in select_best_url: {e}")
            return {"title": "", "url": "", "all_results": []}

def get_esg_markdown(html_content):
    """Parse ESG data from HTML content string (view-source format)"""
    try:
        html_content = re.sub(r'<span[^>]*id="line\d+"[^>]*>.*?</span>', '', html_content, flags=re.DOTALL)
        html_content = re.sub(r'line\d+:', '', html_content)
        
        soup = BeautifulSoup(html_content, 'html.parser')

        # Find the ESG detail table component container
        table_container = soup.find('div', class_=lambda x: x and 'esg-detail-table-component' in x)
        if not table_container:
            return "No esg-detail-table-component found."
        desktop_table = table_container.find('div', class_=lambda x: x and 'esg-table' in x and 'mobile' not in (x or ''))
        
        if desktop_table:
            # Desktop version: structure with rowgroups
            rowgroups = desktop_table.find_all('div', role='rowgroup')
            if len(rowgroups) >= 2:
                # Extract headers
                header_row = rowgroups[0].find('div', role='row')
                headers = [h.get_text(strip=True) for h in header_row.find_all('div', role='columnheader')] if header_row else []
                
                # Extract values
                data_row = rowgroups[1].find('div', role='row')
                values = [v.get_text(strip=True) for v in data_row.find_all('div', role='cell')] if data_row else []
                
                if headers and values:
                    md = "| Field | Value |\n| :--- | :--- |\n"
                    for k, v in zip(headers, values):
                        clean_k = unicodedata.normalize("NFKD", k).replace('&amp;', '&')
                        clean_v = unicodedata.normalize("NFKD", v).replace('&amp;', '&')
                        md += f"| **{clean_k}** | {clean_v} |\n"
                    return md

        mobile_table = table_container.find('div', class_=lambda x: x and 'esg-table-mobile-container' in x)
        
        if mobile_table:
            headers = []
            values = []
            
            all_divs = mobile_table.find_all('div', role=['columnheader', 'cell'])
            for div in all_divs:
                role = div.get('role', '')
                text = div.get_text(strip=True)
                if role == 'columnheader':
                    headers.append(text)
                elif role == 'cell' and text:  
                    values.append(text)
            
            if headers and values:
                md = "| Field | Value |\n| :--- | :--- |\n"
                for k, v in zip(headers, values[:len(headers)]):
                    clean_k = unicodedata.normalize("NFKD", k).replace('&amp;', '&')
                    clean_v = unicodedata.normalize("NFKD", v).replace('&amp;', '&')
                    md += f"| **{clean_k}** | {clean_v} |\n"
                return md

        return "Could not parse ESG table from HTML."

    except Exception as e:
        return f"Error processing HTML: {e}"

async def process_company(company_name, page):
    """
    Process single company using existing browser page
    
    Args:
        company_name: Company name to search
        page: Playwright page object (already open)
    """
    max_retries = 3
    retry_count = 0
    row_data = {}
    
    while retry_count < max_retries:
        if retry_count > 0:
            print(f"\nRetry attempt {retry_count}/{max_retries - 1}...")
        
        if retry_count > 0:
            print(f"(Attempt {retry_count + 1}/{max_retries})")

        selector = SearchAndSelectURL()
        query = f"site:spglobal.com {company_name} ESG Score"

        print(f"Searching: {query}")
        await page.keyboard.press("Control+L")
        await page.keyboard.press("Control+A")
        await page.keyboard.press("Backspace")
        await page.keyboard.type(query, delay=40)
        await page.keyboard.press("Enter")

        print("Waiting for results...")
        await asyncio.sleep(6)

        print("Copying & analyzing results...")
        await page.keyboard.press("Control+A")
        await page.keyboard.press("Control+C")
        await asyncio.sleep(1)

        copied_text = pyperclip.paste()
        
        if not copied_text:
            retry_count += 1
            if retry_count < max_retries:
                await asyncio.sleep(3) 
                continue
            else:
                row_data = {
                    'Company Name': company_name,
                    'URL List': '',
                    'Best Title': '',
                    'Best URL': '',
                    'Result ESG': 'Clipboard empty - search failed (max retries)'
                }
                return
        try:
            result = selector.select_best_url(company_name, copied_text)
        except Exception as e:
            print(f"Error processing search results: {e}")
            retry_count += 1
            if retry_count < max_retries:
                await asyncio.sleep(3)  
                continue
            else:
                row_data = {
                    'Company Name': company_name,
                    'URL List': '',
                    'Best Title': '',
                    'Best URL': '',
                    'Result ESG': f'Error: {str(e)} (max retries)'
                }
                return

        best_title = result.get("title", "")
        best_url = result.get("url", "")
        all_urls = [r["url"] for r in result.get("all_results", []) if r.get("url")]

        if not best_url:
            print(f"No ESG URL found for '{company_name}'!")
            retry_count += 1
            if retry_count < max_retries:
                await asyncio.sleep(3) 
                continue
            else:
                row_data = {
                    'Company Name': company_name,
                    'URL List': '|'.join(all_urls),
                    'Best Title': '',
                    'Best URL': '',
                    'Result ESG': 'No ESG URL found (max retries)'
                }
                return

        await page.keyboard.press("Control+L")
        await asyncio.sleep(0.5) 
        await page.keyboard.press("Control+A")
        await page.keyboard.press("Backspace")
        
        view_source_url = f"view-source:{best_url}"
        print(f"Typing: {view_source_url}")
        await page.keyboard.type(view_source_url, delay=30)
        await asyncio.sleep(1)
        await page.keyboard.press("Enter")

        print("⏳ Loading view-source page...")
        await asyncio.sleep(15) 

        print("Clearing clipboard and copying page content...")
        pyperclip.copy("")
        await asyncio.sleep(0.5)
        
        # Now select all and copy from view-source page
        await page.keyboard.press("Control+A")
        await asyncio.sleep(0.5)
        await page.keyboard.press("Control+C")
        await asyncio.sleep(2)  

        html_content = pyperclip.paste()

        if not html_content:
            esg_result = "Clipboard content too short or empty"
            retry_count += 1
            if retry_count < max_retries:
                await asyncio.sleep(3)
                continue
        else:
            markdown_result = get_esg_markdown(html_content)
            esg_result = markdown_result

        row_data = {
            'Company Name': company_name,
            'URL List': '|'.join(all_urls),
            'Best Title': best_title,
            'Best URL': best_url,
            'Result ESG': esg_result
        }

        if esg_result != "No HTML files found":
            print(f"✨ Successfully processed {company_name}")
            return row_data
        else:
            retry_count += 1
            if retry_count < max_retries:
                print(f"HTML extraction failed, retrying...")
                await asyncio.sleep(3)

    return row_data


# Playwright Browser Initialization

In [9]:
play = await async_playwright().start()

browser = await play.chromium.launch(headless=False)
context = await browser.new_context()
page = await context.new_page()

await page.goto(SEARCH_URL)

<Response url='http://localhost:3000/' request=<Request url='http://localhost:3000/' method='GET'>>

# Crawl data

In [ ]:
companies = [
    "Lasertec Corporation",
    "Shin-Etsu Chemical Co., Ltd.",
    "Mitsui O.S.K. Lines, Ltd.",
    "DeNA Co., Ltd.",
    "CAPCOM CO., LTD.",
    "BRIDGESTONE CORPORATION",
    "DAIICHI SANKYO COMPANY, LIMITED",
    "KAJIMA CORPORATION",
    "JFE Holdings, Inc.",
    "Nintendo Co., Ltd.",
    "Mitsubishi Heavy Industries, Ltd.",
    "ITOCHU Corporation",
    "CASIO COMPUTER CO., LTD.",
    "Bank of Innovation, Inc.",
    "Mitsubishi UFJ Financial Group, Inc.",
    "ADVANTEST CORPORATION",
    "TOYOTA MOTOR CORPORATION",
    "FAST RETAILING CO., LTD.",
    "Mitsui Fudosan Co., Ltd.",
    "SoftBank Group Corp.",
    "KOEI TECMO HOLDINGS CO., LTD.",
    "TOEI ANIMATION CO., LTD.",
    "ENEOS Holdings, Inc.",
    "Mitsubishi Corporation",
    "LY Corporation",
]

for name in companies:
    print("="*20)
    print(f"\n🔄 Processing: {name}")

    row_data = await process_company(name, page)
    
    print(row_data['Result ESG'])

# Browser close

In [7]:
await asyncio.sleep(5)
await browser.close()